# Azure ML notebook: extract data and run default training

This notebook is designed for Azure ML Studio compute. It does not require local workstation files.

Workflow:
1. Install project + ML dependencies in the notebook kernel.
2. Download a global WDI snapshot (1990-2025) directly from World Bank archives.
3. Download the 2026 BoC-BoE default workbook from the official release URL.
4. Run `ml.default_training` and save artifacts.

The resulting artifacts are research outputs only; production prediction serving is a separate step.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Could not find pyproject.toml. Open this notebook from the repository workspace.')

repo = find_repo_root(Path.cwd())
print(f'Repository root: {repo}')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.[ml]'], cwd=str(repo))
print('Dependencies installed.')

In [ ]:
import asyncio
import hashlib
import json
from datetime import datetime, timezone

import httpx

sys.path.insert(0, str(repo))

from ingestion.connectors.world_bank import WorldBankClient
from ingestion.pipelines.global_world_bank import download_global_dataset
from ml.default_training import save_run, train

START_YEAR = 1990
END_YEAR = 2025
OUTPUT_ROOT = repo / 'ml' / 'artifacts' / 'default-onset-azureml'
INPUT_ROOT = repo / 'data' / 'raw'
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

snapshot_path = INPUT_ROOT / f'global-wdi-{START_YEAR}-{END_YEAR}.json'
workbook_dir = INPUT_ROOT / 'global-default'
workbook_dir.mkdir(parents=True, exist_ok=True)
workbook_path = workbook_dir / 'BoC-BoE-Database-2026.xlsx'

print(f'Snapshot path: {snapshot_path}')
print(f'Workbook path: {workbook_path}')
print(f'Output path: {OUTPUT_ROOT}')

In [ ]:
async def build_wdi_snapshot() -> dict:
    async with httpx.AsyncClient(headers={'User-Agent': 'SovereignRiskPlatform/0.1 (economic research)'}) as client:
        connector = WorldBankClient(client, 'https://api.worldbank.org/v2')
        payload = await download_global_dataset(connector, START_YEAR, END_YEAR, None)
    
    observations = payload['observations']
    digest_source = json.dumps(observations, sort_keys=True).encode('utf-8')
    dataset_version = hashlib.sha256(digest_source).hexdigest()

    snapshot = {
        'dataset_version': dataset_version,
        'manifest': {
            'source': payload['source'],
            'start_year': START_YEAR,
            'end_year': END_YEAR,
            'countries': payload['countries'],
            'indicators': payload['indicators'],
            'observations': observations,
            'retrieved_at': datetime.now(timezone.utc).isoformat(),
        },
    }
    return snapshot

snapshot = asyncio.run(build_wdi_snapshot())
snapshot_path.write_text(json.dumps(snapshot, ensure_ascii=True), encoding='utf-8')
print(f'Wrote snapshot: {snapshot_path}')
print(f"Dataset version: {snapshot['dataset_version']}")
print(f"Observation rows: {len(snapshot['manifest']['observations'])}")

In [ ]:
DEFAULT_WORKBOOK_URL = 'https://www.bankofcanada.ca/wp-content/uploads/2026/10/BoC-BoE-Database-2026.xlsx'

if not workbook_path.is_file():
    response = httpx.get(DEFAULT_WORKBOOK_URL, timeout=120)
    response.raise_for_status()
    workbook_path.write_bytes(response.content)
    print(f'Downloaded workbook to: {workbook_path}')
else:
    print(f'Workbook already exists: {workbook_path}')

In [ ]:
report, model, predictions = train(snapshot_path, workbook_path, train_through=2013, validate_through=2018)
save_run(OUTPUT_ROOT, report, model, predictions)

print(f"Selected model: {report['selected_model']}")
print(f"Threshold: {report['threshold_from_validation']:.6f}")
print(f"Artifacts written under: {OUTPUT_ROOT}")

predictions.sort_values('model_probability', ascending=False).head(10)

## Next step (serving)

This notebook produces research artifacts only. To show debt scores on the frontend, the app still needs:
- Backend inference/prediction endpoints wired to model artifacts and stored predictions
- Model status endpoint switched from hardcoded `unavailable` to real model metadata
- Frontend map/risk card bound to real prediction payloads instead of placeholder values